In [5]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np


# Function to parse annotations from RML files
def parse_annotations(rml_file_path):
    annotations = []
    try:
        tree = ET.parse(rml_file_path)
        root = tree.getroot()
        for event in root.findall(".//ns0:Event", namespaces={"ns0": "http://www.respironics.com/PatientStudy.xsd"}):
            family = event.attrib.get("Family")
            type_ = event.attrib.get("Type")
            start = float(event.attrib.get("Start"))
            duration = float(event.attrib.get("Duration"))
            end = start + duration
            annotations.append({"family": family, "type": type_, "start": start, "end": end})
    except Exception as e:
        print(f"Error parsing {rml_file_path}: {e}")
    return annotations

# Function to create spectrograms and labels based on annotations
def create_spectrograms(wav_file_path, annotations, sr=22050, n_mels=128, hop_length=512, target_frames=100):
    spectrograms = []
    labels = []

    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, hop_length=hop_length)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        for annotation in annotations:
            if annotation["family"] == "Respiratory" or annotation["type"] == "Nasal - Snore":
                start_frame = int(annotation["start"] * sr / hop_length)
                end_frame = int(annotation["end"] * sr / hop_length)

                if end_frame > mel_spec_db.shape[1]:
                    continue  # Skip annotations outside the spectrogram's range

                segment = mel_spec_db[:, start_frame:end_frame]

                # Pad or truncate to target_frames
                if segment.shape[1] < target_frames:
                    padding = target_frames - segment.shape[1]
                    segment = np.pad(segment, ((0, 0), (0, padding)), mode="constant")
                elif segment.shape[1] > target_frames:
                    segment = segment[:, :target_frames]

                spectrograms.append(segment)
                labels.append(annotation["type"])
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")

    return np.array(spectrograms), labels

# Function to normalize spectrograms
def normalize_spectrograms(spectrograms):
    spectrograms = np.array(spectrograms)
    spectrograms -= np.min(spectrograms)
    spectrograms /= np.max(spectrograms)
    return spectrograms

# Updated model definition
def build_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        Dropout(0.3),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        Dropout(0.3),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Main function to process folders and train model
def process_and_train_model(folder_paths, sr=22050, n_mels=128, hop_length=512, target_frames=100):
    all_spectrograms = []
    all_labels = []

    for folder_path in folder_paths:
        print(f"Processing folder: {folder_path}")
        wav_files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]
        rml_files = [f for f in os.listdir(folder_path) if f.endswith(".rml")]

        if len(rml_files) != 1:
            print(f"Skipping folder {folder_path}: No unique RML file found.")
            continue

        rml_file_path = os.path.join(folder_path, rml_files[0])

        for wav_file in wav_files:
            wav_file_path = os.path.join(folder_path, wav_file)
            annotations = parse_annotations(rml_file_path)
            spectrograms, labels = create_spectrograms(wav_file_path, annotations, sr, n_mels, hop_length, target_frames)
            if spectrograms.size > 0:
                all_spectrograms.extend(spectrograms)
                all_labels.extend(labels)

    if not all_spectrograms or not all_labels:
        print("No data available for training. Check the file paths and ensure that annotations exist in the .rml files.")
        return

    all_spectrograms = normalize_spectrograms(all_spectrograms)
    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    X_train, X_test, y_train, y_test = train_test_split(all_spectrograms, all_labels_encoded, test_size=0.2, random_state=42)
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    model = build_model((n_mels, target_frames, 1), len(label_encoder.classes_))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32, callbacks=[lr_scheduler])
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

# Specify the folder paths
folder_paths = [
    "/Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/",
    "/Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1010/"
]

# Execute
process_and_train_model(folder_paths)


Processing folder: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/
Processing folder: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1010/
Epoch 1/20
39/39 [==============================] - 5s 69ms/step - loss: 16.5647 - accuracy: 0.5483 - val_loss: 3.2165 - val_accuracy: 0.3571 - lr: 0.0010
Epoch 2/20
39/39 [==============================] - 1s 29ms/step - loss: 1.8607 - accuracy: 0.6718 - val_loss: 23.2098 - val_accuracy: 0.1299 - lr: 0.0010
Epoch 3/20
39/39 [==============================] - 1s 28ms/step - loss: 1.3254 - accuracy: 0.7124 - val_loss: 39.9091 - val_accuracy: 0.1364 - lr: 0.0010
Epoch 4/20
37/39 [===========================>..] - ETA: 0s - loss: 1.1557 - accuracy: 0.7188
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.000500

In [10]:
import os
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def load_rml_annotations(rml_file_path):
    """
    Load annotations from an RML file.
    """
    import xml.etree.ElementTree as ET
    tree = ET.parse(rml_file_path)
    root = tree.getroot()
    annotations = []

    for event in root.findall(".//{http://www.respironics.com/PatientStudy.xsd}Event"):
        family = event.attrib.get("Family", "")
        event_type = event.attrib.get("Type", "")
        start = float(event.attrib.get("Start", "0"))
        duration = float(event.attrib.get("Duration", "0"))
        annotations.append({"family": family, "type": event_type, "start": start, "end": start + duration})
    
    return annotations

def create_spectrograms(wav_file_path, annotations, sr=22050, n_mels=128, hop_length=512, target_frames=128):
    """
    Create spectrograms based on audio and annotations.
    """
    spectrograms, labels = [], []
    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        for annotation in annotations:
            if annotation["type"] in ["ObstructiveApnea", "Hypopnea", "CentralApnea", "MixedApnea", "Snore"]:
                start_frame = int(annotation["start"] * sr)
                end_frame = int(annotation["end"] * sr)
                snippet = audio[start_frame:end_frame]
                mel_spec = librosa.feature.melspectrogram(y=snippet, sr=sr, n_mels=n_mels, hop_length=hop_length)
                mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
                if mel_spec_db.shape[1] >= target_frames:
                    spectrograms.append(mel_spec_db[:, :target_frames])
                    labels.append(annotation["type"])
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")
    return np.array(spectrograms), labels

def process_and_train_model(folder_paths, sr=22050, n_mels=128, hop_length=512, target_frames=128):
    """
    Process audio and annotations to train a model.
    """
    all_spectrograms, all_labels = [], []
    label_map = {"ObstructiveApnea": 0, "Hypopnea": 1, "CentralApnea": 2, "MixedApnea": 3, "Snore": 4}

    for folder_path in folder_paths:
        print(f"Processing folder: {folder_path}")
        rml_files = [f for f in os.listdir(folder_path) if f.endswith('.rml')]
        wav_files = [f for f in os.listdir(folder_path) if f.endswith('_cleaned.wav')]

        for rml_file in rml_files:
            rml_file_path = os.path.join(folder_path, rml_file)
            annotations = load_rml_annotations(rml_file_path)
            for wav_file in wav_files:
                wav_file_path = os.path.join(folder_path, wav_file)
                print(f"Using annotations from {rml_file_path} for .wav file: {wav_file_path}")
                spectrograms, labels = create_spectrograms(wav_file_path, annotations, sr, n_mels, hop_length, target_frames)
                all_spectrograms.extend(spectrograms)
                all_labels.extend([label_map[label] for label in labels])

    if len(all_spectrograms) == 0 or len(all_labels) == 0:
        print("No data available for training. Check the file paths and ensure that annotations exist in the .rml files.")
        return

    all_spectrograms = np.array(all_spectrograms)
    all_labels = np.array(all_labels)
    all_spectrograms = all_spectrograms[..., np.newaxis]

    X_train, X_test, y_train, y_test = train_test_split(all_spectrograms, all_labels, test_size=0.2, random_state=42)

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(n_mels, target_frames, 1)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(len(label_map), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)
    y_pred = np.argmax(model.predict(X_test), axis=1)
    print(classification_report(y_test, y_pred, target_names=label_map.keys()))

# Specify folder paths
folder_paths = [
    "/Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/",
    "/Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1010/"
]

# Execute
process_and_train_model(folder_paths)


Processing folder: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/
Using annotations from /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/filtered_00001057-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/00001057-100507_snore_cleaned.wav
Using annotations from /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8/Team Project/Data Preprocessing/1057/filtered_00001057-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/GoogleDrive-tarlan.sultanov01@gmail.com/.shortcut-targets-by-id/1FAWJCPKuxjfQ4HhAXFQk6-_8mbaATyK8